In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

from ksw import Cosmology, utils
from ksw import estimator
import camb
import healpy as hp

from ksw import Afunctionals

In [2]:
# Setup CAMB parameters
pars = camb.CAMBparams()
pars.WantTensors=True
pars.set_cosmology(H0=67.66, ombh2=0.02242, omch2=0.11933)
pars.InitPower.set_params(As=2.1056e-9, ns=0.9665, r=0.001)
cosmo = Cosmology(pars, verbose=False)

# Compute transfer functions
print("Computing transfer functions...")
cosmo.compute_transfer(lmax=300)

# Compute angular power spectra
print("Computing angular power spectra...")
#cosmo.compute_c_ell()

Computing transfer functions...
Computing angular power spectra...


In [3]:
from ksw.shape import Shape
# radii = np.logspace(0, 3, 10)  # Small number of radii for quick testing
radii = np.asarray([10000, 10100])

prim_shape = Shape.prim_local(ns=0.9665)
#radii = np.logspace(0, 3, 10)

cosmo.compute_transfer(lmax=300)
cosmo.add_prim_reduced_bispectrum_scalar_dL(prim_shape, radii)

cosmo.compute_transfer_tensor(lmax=300)
cosmo.add_prim_reduced_bispectrum_tensor_dL(prim_shape, radii)

Updated CAMB param: WantTensors from False to True.
Updated CAMB param: WantTensors from True to False.


In [4]:
# Create a estimator instance
red_bispectra = cosmo.red_bispectra
icov = lambda alm: alm
pol = ('T', 'E', 'B')
lmax = 5
estimator_con = estimator.KSW(red_bispectra, icov, lmax=lmax, pol=pol, precision='double')

In [5]:
import healpy as hp
lmax = estimator_con.lmax
nelem = hp.Alm.getsize(lmax+1)
print(nelem)

28


In [23]:
# Generate sample alm data for testing
lmax = estimator_con.lmax
print(f'{lmax=}')
npol = 3
#nelem = (lmax + 1) * (lmax + 2) // 2   # HEALPix alm element count
nelem = hp.Alm.getsize(lmax)

# Create random alm with correct shape
np.random.seed(20)
#alm_test = np.random.randn(npol, nelem) + 1j * np.random.randn(npol, nelem)
cls = np.zeros((4, lmax + 1))
cls[0] = np.ones(lmax + 1)
cls[1] = np.ones(lmax + 1)
cls[2] = np.ones(lmax + 1)
alm_test = hp.synalm((cls[0], cls[1], cls[2], cls[3]), lmax=lmax)
print(f'{alm_test[0,0]=}')
print(f'{alm_test.shape=}')
print(f'{alm_test.dtype=}')

# Try to call compute_estimate_sst

Lmax = lmax
L_list = np.arange(Lmax + 1)

print(alm_test.dtype)
print(L_list.dtype)

estimate, cubic, lin_term, fisher = estimator_con.compute_estimate_sst(
    alm_test,
    L_list,
    Lmax,
    theta_batch=25
)

lmax=5
alm_test[0,0]=np.complex128(0.8838931126173458+0j)
alm_test.shape=(3, 21)
alm_test.dtype=dtype('complex128')
complex128
int64
(4, 3, 2, 6)
0.0
com_idx=0
1 -5.082198e-21 2.223810e-01
0 1.985233e-23 1.012285e-01
7 -4.632211e-23 1.012285e-01
2 -4.743385e-20 3.137066e-01
3 -2.846031e-19 3.626838e-01
4 -2.846031e-19 3.626838e-01
5 -4.743385e-20 3.137066e-01
6 -4.235165e-21 2.223810e-01
Afunc_product=-2.3827714563007504e-19
vals[0]=np.float64(0.447213595499958)
com_idx=1
2 -1.220412e-06 3.137066e-01
5 -1.220412e-06 3.137066e-01
6 -7.919549e-08 2.223810e-01
1 -7.919549e-08 2.223810e-01
3 -3.877355e-06 3.626838e-01
4 -3.877355e-06 3.626838e-01
0 -1.632607e-10 1.012285e-01
7 -1.632607e-10 1.012285e-01
Afunc_product=-3.6134667798157627e-06
vals[0]=np.float64(-0.31622776601683794)
com_idx=2
0 -2.504160e-26 1.012285e-01
6 4.859686e-24 2.223810e-01
1 -1.033976e-24 2.223810e-01
3 7.940934e-23 3.626838e-01
7 -3.231174e-27 1.012285e-01
2 3.308722e-24 3.137066e-01
5 3.308722e-24 3.137066e-01
4 7